# SAM 3.1 resistor-body COCO rebuild + merge

This notebook has two parts:

1. **Single-image SAM 3.1 test first** — upload one image and verify the single highest-confidence `resistor body` mask.
2. **Full dataset rebuild** — download the three COCO resistor datasets from the `resistor_model` `r1` release, run the same `resistor body` text prompt once per image, assign exactly one SAM body mask to each old resistor annotation, remove images with no accepted body mask, deduplicate exact duplicate images, and merge/split the result 80/10/10.

Use a GPU runtime. Visible progress is printed throughout the full workflow.

**Checkpoint:** this notebook uses the open `AEmotionStudio/sam3.1` mirror and downloads `sam3.1_multiplex.pt` (~3.5 GB) automatically. No Meta/Hugging Face access request is needed. You can also set `SAM3_CHECKPOINT_PATH` to an existing local copy.

**Inference dtype:** SAM 3 image inference is run inside CUDA BF16 autocast. On Tesla T4 Flash Attention remains disabled, but the dtype wrapper prevents the upstream BF16/FP32 mismatch.

## 1. Bootstrap SAM 3.1 for the single-image test

Run this cell first. It installs the official SAM 3 code and downloads/loads the open `AEmotionStudio/sam3.1` multiplex checkpoint only; it does **not** download any resistor dataset.

In [ ]:
import urllib.request
print('[launcher] downloading SAM 3.1 single-image bootstrap...', flush=True)
url = 'https://raw.githubusercontent.com/Persie0/resistor_model/main/colab/sam3_test_bootstrap.py'
path = '/content/sam3_test_bootstrap.py'
urllib.request.urlretrieve(url, path)
exec(compile(open(path, encoding='utf-8').read(), path, 'exec'))


## 2. Test the selected resistor-body mask on one image

Upload one resistor image. SAM 3.1 runs the text prompt `resistor body`, selects the returned candidate with the highest confidence, and renders **only that mask**. This is the same body-only signal used by the full COCO rebuild.

In [ ]:
import io
import time
import matplotlib.pyplot as plt
from google.colab import files

if not globals().get('SAM3_BOOTSTRAPPED', False):
    raise RuntimeError('Run the SAM 3.1 bootstrap cell above first.')

print('[test 1/4] Upload one image...', flush=True)
uploaded_test = files.upload()
if not uploaded_test:
    raise RuntimeError('No image uploaded.')

test_name, test_bytes = next(iter(uploaded_test.items()))
print(f'[test 2/4] Loading {test_name} ({len(test_bytes) / 1024:.1f} KiB)...', flush=True)
test_image = Image.open(io.BytesIO(test_bytes)).convert('RGB')
print(f'[test] image size: {test_image.width}x{test_image.height}', flush=True)

print('[test 3/4] Running SAM 3.1 with prompt: resistor body (BF16 autocast)', flush=True)
test_started = time.monotonic()
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    test_state = processor.set_image(test_image)
    test_out = processor.set_text_prompt(prompt='resistor body', state=test_state)
test_elapsed = time.monotonic() - test_started

test_masks_tensor = test_out['masks']
if test_masks_tensor.ndim == 4 and test_masks_tensor.shape[1] == 1:
    test_masks_tensor = test_masks_tensor.squeeze(1)
test_masks = test_masks_tensor.detach().cpu().numpy().astype(np.uint8)
test_boxes = test_out['boxes'].detach().float().cpu().numpy()
test_scores = test_out['scores'].detach().float().cpu().numpy().astype(float)

if len(test_scores) == 0:
    raise RuntimeError('SAM 3.1 returned no resistor-body candidates for this image.')

test_best_index = int(np.argmax(test_scores))
test_best_mask = test_masks[test_best_index]
test_best_box = test_boxes[test_best_index]
test_best_score = float(test_scores[test_best_index])

print(f'[test] inference complete in {test_elapsed:.2f}s | raw candidates: {len(test_scores)}', flush=True)
print(f'[test] selected body candidate {test_best_index}: score={test_best_score:.4f}, box={test_best_box.tolist()}', flush=True)

print('[test 4/4] Rendering selected body mask only...', flush=True)
base = np.array(test_image).copy()
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(base)
overlay = np.zeros((*test_best_mask.shape, 4), dtype=np.float32)
overlay[..., 3] = test_best_mask.astype(np.float32) * 0.35
overlay[..., 0] = test_best_mask.astype(np.float32)
ax.imshow(overlay)

x1, y1, x2, y2 = map(float, test_best_box)
rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2)
ax.add_patch(rect)
ax.text(x1, max(0, y1 - 5), f'{test_best_index}: {test_best_score:.3f}', fontsize=10, backgroundcolor='white')
ax.set_title(f'SAM 3.1 selected resistor body: {test_name}')
ax.axis('off')
plt.show()
print('[test] done', flush=True)


## 3. Build the merged body-only COCO dataset

The full workflow uses `resistor body` once per image and matches returned body masks one-to-one against the old COCO resistor instances. Each old instance gets at most one SAM mask and each SAM mask can be used only once. The selected SAM segmentation replaces the old annotation; images with zero accepted masks are omitted.

In [ ]:
import gc
import os
import urllib.request
import huggingface_hub

if not globals().get('SAM3_BOOTSTRAPPED', False):
    raise RuntimeError('Run the SAM 3.1 bootstrap successfully before starting the full dataset workflow.')

sam3_checkpoint_for_full = globals().get('sam3_checkpoint')
if not sam3_checkpoint_for_full:
    raise RuntimeError('Bootstrap did not expose a resolved SAM 3.1 checkpoint path.')

os.environ['SAM3_CHECKPOINT_PATH'] = sam3_checkpoint_for_full
huggingface_hub.notebook_login = lambda *args, **kwargs: print('[auth] skipped: AEmotionStudio/sam3.1 is public', flush=True)
huggingface_hub.login = lambda *args, **kwargs: None
import sam3.model_builder as sam3_model_builder
sam3_model_builder.download_ckpt_from_hf = lambda version='sam3': sam3_checkpoint_for_full

print(f'[launcher] reusing AEmotionStudio SAM 3.1 checkpoint: {sam3_checkpoint_for_full}', flush=True)
print('[launcher] keeping the loaded SAM 3.1 model for the body-only bulk workflow', flush=True)
try:
    del test_state
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('[launcher] downloading latest body-only SAM merge workflow...', flush=True)
url = 'https://raw.githubusercontent.com/Persie0/resistor_model/main/colab/sam3_merge_3_resistor_datasets.py'
path = '/content/sam3_merge_3_resistor_datasets.py'
urllib.request.urlretrieve(url, path)
print('[launcher] starting three-dataset body-only COCO rebuild...', flush=True)
exec(compile(open(path, encoding='utf-8').read(), path, 'exec'))
